# Week 2

1. Prompt caching
2. Multi-turn conversation between two models
3. Chat history and the system prompt
4. Multimodal
5. Tools
6. Prototype: Gradio UI, streaming, expert system prompt, model switch, tool calling

In [ ]:
import base64
import json
import os
from io import BytesIO
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from PIL import Image, ImageDraw

In [ ]:
for folder in [Path.cwd(), *Path.cwd().parents]:
    env_file = folder / ".env"
    if env_file.exists():
        load_dotenv(env_file, override=True)
        break

openai = OpenAI()
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

MODEL_GPT = "gpt-4o-mini"
MODEL_GEMINI = "gemini-3.1-flash-lite"
MODEL_OLLAMA = "qwen2.5:0.5b"

## 1. Prompt caching

A long static prefix placed first, variable content last. The provider can then reuse the prefix across calls and report it as cached input tokens.

In [ ]:
notes = (
    "Reference notes.\n"
    + ("yield from delegates to a sub-iterator. A set comprehension de-duplicates. " * 400)
)


def ask_with_prefix(question):
    return gemini.chat.completions.create(
        model=MODEL_GEMINI,
        messages=[
            {"role": "system", "content": notes},
            {"role": "user", "content": question},
        ],
        max_tokens=80,
    )


def show_usage(label, response):
    usage = response.usage
    details = getattr(usage, "prompt_tokens_details", None)
    cached = getattr(details, "cached_tokens", 0) or 0 if details else 0
    print(f"{label}: prompt={usage.prompt_tokens} cached={cached}")


show_usage("call 1", ask_with_prefix("One sentence: what does yield from do?"))
show_usage("call 2", ask_with_prefix("One sentence: why use a set comprehension?"))

## 2. Multi-turn conversation between two models

Each model sees its own lines as `assistant` and the other model's lines as `user`.

In [ ]:
gemini_system = "You are argumentative and snarky. Disagree. Two short sentences."
ollama_system = "You are polite. Find common ground. Two short sentences."

gemini_lines = ["Hi there"]
ollama_lines = ["Hi"]


def call_gemini_bot():
    messages = [{"role": "system", "content": gemini_system}]
    for mine, theirs in zip(gemini_lines, ollama_lines):
        messages.append({"role": "assistant", "content": mine})
        messages.append({"role": "user", "content": theirs})
    return gemini.chat.completions.create(
        model=MODEL_GEMINI, messages=messages, max_tokens=80
    ).choices[0].message.content


def call_ollama_bot():
    messages = [{"role": "system", "content": ollama_system}]
    for theirs, mine in zip(gemini_lines, ollama_lines):
        messages.append({"role": "user", "content": theirs})
        messages.append({"role": "assistant", "content": mine})
    messages.append({"role": "user", "content": gemini_lines[-1]})
    return ollama.chat.completions.create(
        model=MODEL_OLLAMA, messages=messages, max_tokens=80
    ).choices[0].message.content


display(Markdown(f"**Gemini:** {gemini_lines[0]}\n\n**Ollama:** {ollama_lines[0]}"))
for _ in range(3):
    reply = call_gemini_bot()
    gemini_lines.append(reply)
    display(Markdown(f"**Gemini:** {reply}"))

    reply = call_ollama_bot()
    ollama_lines.append(reply)
    display(Markdown(f"**Ollama:** {reply}"))

## 3. Chat history and the system prompt

The callback Gradio expects. Every turn rebuilds `system + history + new message`, which is what makes the model appear to remember.

In [ ]:
def build_messages(system, history, message):
    return (
        [{"role": "system", "content": system}]
        + [{"role": h["role"], "content": h["content"]} for h in history]
        + [{"role": "user", "content": message}]
    )


system = "You are terse. Answer in one short sentence."
history = []

for turn in ["My favourite language is Python.", "What is a generator?", "What was my favourite language?"]:
    messages = build_messages(system, history, turn)
    response = openai.chat.completions.create(
        model=MODEL_GPT, messages=messages, max_tokens=60
    )
    reply = response.choices[0].message.content
    history += [
        {"role": "user", "content": turn},
        {"role": "assistant", "content": reply},
    ]
    print(f"messages sent={len(messages)} prompt_tokens={response.usage.prompt_tokens}")
    display(Markdown(f"**{turn}**\n\n{reply}"))

## 4. Multimodal

An image is passed as a base64 data URL inside the user message content list.

In [ ]:
img = Image.new("RGB", (256, 256), "navy")
ImageDraw.Draw(img).ellipse((48, 48, 208, 208), fill="gold")
display(img)

buffer = BytesIO()
img.save(buffer, format="PNG")
data_url = f"data:image/png;base64,{base64.b64encode(buffer.getvalue()).decode()}"

vision = gemini.chat.completions.create(
    model=MODEL_GEMINI,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe this image in one sentence."},
            {"type": "image_url", "image_url": {"url": data_url}},
        ],
    }],
    max_tokens=80,
)
display(Markdown(vision.choices[0].message.content))

## 5. Tools

The model does not run the function. It replies with `finish_reason == "tool_calls"` to ask us to run it, and we send the result back as a `tool` message.

In [ ]:
PYTHON_DOCS = {
    "yield from": "https://docs.python.org/3/reference/expressions.html#yield-expressions",
    "set comprehension": "https://docs.python.org/3/tutorial/datastructures.html#sets",
    "generator": "https://docs.python.org/3/glossary.html#term-generator",
    "decorator": "https://docs.python.org/3/glossary.html#term-decorator",
}


def lookup_docs(topic):
    print(f"Tool called for topic {topic}")
    url = PYTHON_DOCS.get(topic.lower(), "No official docs entry found")
    return f"Official docs for {topic}: {url}"


docs_function = {
    "name": "lookup_docs",
    "description": "Get the official Python documentation URL for a language topic.",
    "parameters": {
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "The Python topic to look up, such as 'yield from'",
            },
        },
        "required": ["topic"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": docs_function}]


def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "lookup_docs":
            arguments = json.loads(tool_call.function.arguments)
            responses.append({
                "role": "tool",
                "content": lookup_docs(arguments.get("topic")),
                "tool_call_id": tool_call.id,
            })
    return responses

In [ ]:
messages = [
    {"role": "system", "content": "Cite the official docs link when you explain something."},
    {"role": "user", "content": "Explain yield from and link the official docs."},
]

response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)

while response.choices[0].finish_reason == "tool_calls":
    reply = response.choices[0].message
    messages.append(reply)
    messages.extend(handle_tool_calls(reply))
    response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)

display(Markdown(response.choices[0].message.content))

## 6. Prototype

The Week 1 explainer rebuilt with everything above: a Gradio UI, streaming, expertise set through the system prompt, a model switch, and the docs tool.

Tool calls are resolved first, then the final answer is streamed. The local model does not get tools.

In [ ]:
import gradio as gr

EXPERT = """You are a senior software engineer explaining code to another engineer.
Be precise and concise. Name the mechanism, then give a short example, then the pitfall.
Call lookup_docs when an official reference would help.
Respond in markdown. Do not wrap the whole reply in a code fence."""

MODELS = {
    "gpt-4o-mini": (openai, MODEL_GPT, True),
    "gemini-3.1-flash-lite": (gemini, MODEL_GEMINI, True),
    "qwen2.5:0.5b (local)": (ollama, MODEL_OLLAMA, False),
}


def chat(message, history, model_name):
    client, model, supports_tools = MODELS[model_name]
    messages = build_messages(EXPERT, history, message)

    if supports_tools:
        response = client.chat.completions.create(
            model=model, messages=messages, tools=tools
        )
        while response.choices[0].finish_reason == "tool_calls":
            reply = response.choices[0].message
            messages.append(reply)
            messages.extend(handle_tool_calls(reply))
            response = client.chat.completions.create(
                model=model, messages=messages, tools=tools
            )

    stream = client.chat.completions.create(
        model=model, messages=messages, stream=True
    )
    answer = ""
    for chunk in stream:
        answer += chunk.choices[0].delta.content or ""
        yield answer


gr.ChatInterface(
    fn=chat,
    type="messages",
    additional_inputs=[
        gr.Radio(list(MODELS), value="gpt-4o-mini", label="model")
    ],
).launch(inbrowser=True)